**Date Exploration**

In [ ]:
# import libraries
import pandas as pd
from sqlalchemy import create_engine
import psycopg2

In [ ]:
df = pd.read_csv("Walmart.csv")

In [ ]:
df.shape

(10051, 11)

In [ ]:
df.head()

,invoice_id,Branch,City,category,unit_price,quantity,date,time,payment_method,rating,profit_margin
0,1,WALM003,San Antonio,Health and beauty,$74.69,7.0,05/01/19,13:08:00,Ewallet,9.1,0.48
1,2,WALM048,Harlingen,Electronic accessories,$15.28,5.0,08/03/19,10:29:00,Cash,9.6,0.48
2,3,WALM067,Haltom City,Home and lifestyle,$46.33,7.0,03/03/19,13:23:00,Credit card,7.4,0.33
3,4,WALM064,Bedford,Health and beauty,$58.22,8.0,27/01/19,20:33:00,Ewallet,8.4,0.33
4,5,WALM013,Irving,Sports and travel,$86.31,7.0,08/02/19,10:37:00,Ewallet,5.3,0.48


In [ ]:
df.describe(include="all")

,invoice_id,Branch,City,category,unit_price,quantity,date,time,payment_method,rating,profit_margin
count,10051.000000,10051,10051,10051,10020,10020.000000,10051,10051,10051,10051.000000,10051.000000
unique,NaN,100,98,6,1008,NaN,1460,1001,3,NaN,NaN
top,NaN,WALM058,Weslaco,Fashion accessories,$63,NaN,01/12/21,15:48:00,Credit card,NaN,NaN
freq,NaN,240,399,4579,159,NaN,48,33,4260,NaN,NaN
mean,5025.741220,NaN,NaN,NaN,NaN,2.353493,NaN,NaN,NaN,5.825659,0.393791
std,2901.174372,NaN,NaN,NaN,NaN,1.602658,NaN,NaN,NaN,1.763991,0.090669
min,1.000000,NaN,NaN,NaN,NaN,1.000000,NaN,NaN,NaN,3.000000,0.180000
25%,2513.500000,NaN,NaN,NaN,NaN,1.000000,NaN,NaN,NaN,4.000000,0.330000
50%,5026.000000,NaN,NaN,NaN,NaN,2.000000,NaN,NaN,NaN,6.000000,0.330000
75%,7538.500000,NaN,NaN,NaN,NaN,3.000000,NaN,NaN,NaN,7.000000,0.480000


In [ ]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10051 entries, 0 to 10050
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   invoice_id      10051 non-null  int64  
 1   Branch          10051 non-null  str    
 2   City            10051 non-null  str    
 3   category        10051 non-null  str    
 4   unit_price      10020 non-null  str    
 5   quantity        10020 non-null  float64
 6   date            10051 non-null  str    
 7   time            10051 non-null  str    
 8   payment_method  10051 non-null  str    
 9   rating          10051 non-null  float64
 10  profit_margin   10051 non-null  float64
dtypes: float64(3), int64(1), str(7)
memory usage: 863.9 KB


In [ ]:
df.duplicated().sum()

np.int64(51)

In [ ]:
df.isnull().sum()

invoice_id         0
Branch             0
City               0
category           0
unit_price        31
quantity          31
date               0
time               0
payment_method     0
rating             0
profit_margin      0
dtype: int64

**Data Cleaning**

In [ ]:
# drop duplicates
df.drop_duplicates(inplace=True)
df.duplicated().sum()

np.int64(0)

In [ ]:
# check for nulls
df[df["quantity"].isnull()][["quantity", "unit_price"]]

,quantity,unit_price
1892,NaN,NaN
1893,NaN,NaN
1894,NaN,NaN
1895,NaN,NaN
1896,NaN,NaN
1897,NaN,NaN
1898,NaN,NaN
1899,NaN,NaN
1900,NaN,NaN
1901,NaN,NaN


In [ ]:
# drop nulls
df.dropna(inplace=True)
df.isnull().sum()

invoice_id        0
Branch            0
City              0
category          0
unit_price        0
quantity          0
date              0
time              0
payment_method    0
rating            0
profit_margin     0
dtype: int64

In [ ]:
# handle "unit_price" column data type
df["unit_price"] = df["unit_price"].str.replace("$", "").astype(float)
df["unit_price"].head()

AttributeError: Can only use .str accessor with string values, not floating

In [ ]:
# add "total_price" column
df["total_price"] = df["unit_price"] * df["quantity"]
df["total_price"].head()

0    522.83
1     76.40
2    324.31
3    465.76
4    604.17
Name: total_price, dtype: float64

In [ ]:
df.info()

<class 'pandas.DataFrame'>
Index: 9969 entries, 0 to 9999
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   invoice_id      9969 non-null   int64  
 1   Branch          9969 non-null   str    
 2   City            9969 non-null   str    
 3   category        9969 non-null   str    
 4   unit_price      9969 non-null   float64
 5   quantity        9969 non-null   float64
 6   date            9969 non-null   str    
 7   time            9969 non-null   str    
 8   payment_method  9969 non-null   str    
 9   rating          9969 non-null   float64
 10  profit_margin   9969 non-null   float64
 11  total_price     9969 non-null   float64
dtypes: float64(5), int64(1), str(6)
memory usage: 1012.5 KB


In [ ]:
# save file
df.to_csv("Walmart_clean_data.csv", index=False)

In [ ]:
df.shape

(9969, 12)

In [ ]:
engine_sql = create_engine(
    "mssql+pyodbc://@localhost/WalmartSales?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes")

In [ ]:
df.to_sql("Walmart", con=engine_sql, if_exists="replace", index=False)

51

In [ ]:
# This asks the database directly: "How many rows do you have?"
pd.read_sql("SELECT COUNT(*) AS total_rows FROM Walmart", engine_sql)

,total_rows
0,9969
